# Explicabilidad de préstamos

Entrenamos un Random Forest y exploramos sus predicciones con SHAP. Ejecuta las celdas en orden desde VS Code con la extensión de Colab.

## Librerías

Instalamos las herramientas del laboratorio, incluido LIME para las siguientes etapas.

In [ ]:
%pip install -q pandas scikit-learn "shap>=0.45" lime matplotlib

## Etapa 2: Cargar y explorar los datos

`Loan_Status` indica si el préstamo fue aprobado (`Y`) o denegado (`N`). Revisamos los datos faltantes y quitamos `Loan_ID`, que solo identifica cada solicitud.

In [ ]:
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

url = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
df = pd.read_csv(url)
display(df.head())
print("Datos faltantes por columna:\n", df.isna().sum())
df = df.drop(columns="Loan_ID")

## Etapa 3: Preparar los datos

Usamos el 80 % para entrenar y el 20 % para probar; la semilla 42 permite repetir la división. Rellenamos números con la mediana y categorías con la moda del entrenamiento, y convertimos cada categoría en una columna de 0 y 1.

In [ ]:
X = df.drop(columns="Loan_Status")
y = df["Loan_Status"].map({"Y": 1, "N": 0})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

# Aprendemos los valores de relleno solo del entrenamiento.
relleno = X_train[num_cols].median().to_dict()
relleno.update(X_train[cat_cols].mode().iloc[0].to_dict())
X_train = X_train.fillna(relleno)
X_test = X_test.fillna(relleno)

X_train_prep = pd.get_dummies(X_train, columns=cat_cols, dtype=int)
X_test_prep = pd.get_dummies(X_test, columns=cat_cols, dtype=int)
# Usamos exactamente las columnas del entrenamiento, en el mismo orden.
X_test_prep = X_test_prep.reindex(columns=X_train_prep.columns, fill_value=0)

print("Entrenamiento:", X_train_prep.shape)
print("Prueba:", X_test_prep.shape)

## Etapa 4: Entrenar y evaluar

Accuracy mide la proporción total de aciertos. F1 combina precisión y capacidad de detectar los préstamos aprobados (clase 1); ambas métricas van de 0 a 1 y los valores más altos son mejores.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_prep, y_train)
y_pred = rf_model.predict(X_test_prep)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score: {f1:.4f}")

## Etapa 5: Explicación global con SHAP

Explicamos la clase aprobado (1): cada punto es una solicitud y las variables más influyentes aparecen arriba. Los valores SHAP positivos favorecen la aprobación y los negativos la reducen; rojo indica valores altos de la variable y azul, bajos.

In [ ]:
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer(X_test_prep)
clase_aprobado = list(rf_model.classes_).index(1)
shap_values_aprobado = shap_values[:, :, clase_aprobado]

shap.plots.beeswarm(shap_values_aprobado, show=False)
plt.title("SHAP: clase aprobado")
plt.show()